# Exploration des fichiers clients bruts (C10)

Documente pas à pas comment les chiffres cités dans
[`docs/architecture/topographie_donnees.md`](../docs/architecture/topographie_donnees.md) §3.3
et implémentés dans
[`src/datacore/processing/clients_cleaning.py`](../src/datacore/processing/clients_cleaning.py)
ont été obtenus, pour que l'analyse reste vérifiable et reproductible
plutôt que de rester une exécution ponctuelle non tracée.

**Contexte** : trois fichiers de commandes brutes reçues des clients
NordDrive, FreshMarket et MedioTex (`data/raw/clients_fichiers/`),
volontairement hétérogènes et imparfaits (cahier des charges, §3.1).


In [1]:
import csv
from collections import Counter

FILES = {
    "norddrive": (
        "../data/raw/clients_fichiers/norddrive_commandes.csv",
        ";", "ref_commande", "reference_piece",
    ),
    "freshmarket": (
        "../data/raw/clients_fichiers/freshmarket_commandes.csv",
        ",", "id_commande_client", "code_article",
    ),
    "mediotex": (
        "../data/raw/clients_fichiers/mediotex_commandes.csv",
        ",", "numero_cde", "sku",
    ),
}

rows_by_client = {
    name: list(csv.DictReader(open(path, encoding="utf-8"), delimiter=sep))
    for name, (path, sep, _, _) in FILES.items()
}

for name, rows in rows_by_client.items():
    print(name, ":", len(rows), "lignes")


norddrive : 1479 lignes
freshmarket : 1288 lignes
mediotex : 1496 lignes


## 1. Le piège du grain "commande" seul

Un premier réflexe consiste à mesurer les doublons sur l'identifiant de
commande seul. C'est ce qui a été fait dans la toute première version de
la topographie des données — et c'est trompeur, comme le montre l'exemple
ci-dessous.


In [2]:
name = "norddrive"
path, sep, key, prod_key = FILES[name]
rows = rows_by_client[name]

counts = Counter(r[key] for r in rows)
dup_id = next(k for k, v in counts.items() if v > 1)
print(f"Exemple : {dup_id} apparait {counts[dup_id]} fois")
for r in [r for r in rows if r[key] == dup_id]:
    print(" ", r)


Exemple : ND-000549 apparait 4 fois
  {'ref_commande': 'ND-000549', 'date_cde': '19-07-2026', 'reference_piece': 'SKU-10005', 'designation': "Bougie d'allumage", 'qte': '17', 'poids_unitaire_g': '5900', 'entrepot': 'OMG-LIL'}
  {'ref_commande': 'ND-000549', 'date_cde': '19-07-2026', 'reference_piece': 'SKU-10004', 'designation': 'Filtre a air', 'qte': '40', 'poids_unitaire_g': '1820', 'entrepot': 'OMG-LIL'}
  {'ref_commande': 'ND-000549', 'date_cde': '19-07-2026', 'reference_piece': 'SKU-10003', 'designation': 'Filtre a huile', 'qte': '27', 'poids_unitaire_g': '2240', 'entrepot': 'OMG-LIL'}
  {'ref_commande': 'ND-000549', 'date_cde': '2026-07-19', 'reference_piece': 'SKU-10003', 'designation': 'Filtre a huile', 'qte': '', 'poids_unitaire_g': '2240', 'entrepot': 'OMG-LIL'}


On voit immédiatement que ces 4 lignes portent la **même date** et le
**même entrepôt**, mais des `reference_piece` (SKU) et quantités
**différentes** : ce n'est pas un doublon, c'est une commande
multi-produits légitime (une ligne par produit commandé), exactement
comme `lignes_commande` dans FluxPro. Mesurer les "doublons" à ce grain
revient à compter le nombre moyen de produits par commande, pas la
qualité des données.


In [3]:
for name, (path, sep, key, prod_key) in FILES.items():
    rows = rows_by_client[name]
    counts = Counter(r[key] for r in rows)
    n_repeated_ids = sum(1 for v in counts.values() if v > 1)
    print(f"{name}: {len(rows)} lignes, {len(counts)} commandes uniques, "
          f"{n_repeated_ids} commandes avec plusieurs lignes "
          f"({n_repeated_ids / len(counts):.0%} des commandes)")


norddrive: 1479 lignes, 466 commandes uniques, 372 commandes avec plusieurs lignes (80% des commandes)
freshmarket: 1288 lignes, 441 commandes uniques, 354 commandes avec plusieurs lignes (80% des commandes)
mediotex: 1496 lignes, 493 commandes uniques, 388 commandes avec plusieurs lignes (79% des commandes)


C'est ce calcul (au grain commande seul) qui produisait le chiffre erroné
de "55 à 68 % de doublons" dans la première version de la topographie des
données — en réalité le taux de commandes multi-produits, pas un taux de
doublons.


## 2. Le bon grain : (commande, produit)

Le référentiel FluxPro modélise une commande comme `commandes` +
`lignes_commande` (une ligne par produit). Le grain équivalent pour les
fichiers clients bruts est donc **(commande, produit)** : c'est à ce
niveau qu'un doublon a un sens (la même ligne de commande apparaissant
plusieurs fois).


In [4]:
for name, (path, sep, key, prod_key) in FILES.items():
    rows = rows_by_client[name]
    grain_counts = Counter((r[key], r[prod_key]) for r in rows)
    true_dups = {k: v for k, v in grain_counts.items() if v > 1}
    print(f"{name}: {len(rows)} lignes, {len(grain_counts)} couples (commande, produit) uniques, "
          f"{len(true_dups)} vrais doublons ({len(true_dups) / len(grain_counts):.1%})")


norddrive: 1479 lignes, 1263 couples (commande, produit) uniques, 186 vrais doublons (14.7%)
freshmarket: 1288 lignes, 1110 couples (commande, produit) uniques, 161 vrais doublons (14.5%)
mediotex: 1496 lignes, 1260 couples (commande, produit) uniques, 218 vrais doublons (17.3%)


Ce sont ces pourcentages (**13 à 17 %** selon le client) qui sont retenus
dans la topographie des données et dans le rapport de nettoyage de
`clean_and_aggregate()`.

### Exemple de vrai doublon, avec conflit de valeurs


In [5]:
name = "norddrive"
path, sep, key, prod_key = FILES[name]
rows = rows_by_client[name]
grain_counts = Counter((r[key], r[prod_key]) for r in rows)
true_dups = {k: v for k, v in grain_counts.items() if v > 1}

example_key = next(iter(true_dups))
print("Doublon", example_key, ":")
for r in rows:
    if (r[key], r[prod_key]) == example_key:
        print(" ", r)


Doublon ('ND-001297', 'SKU-10008') :
  {'ref_commande': 'ND-001297', 'date_cde': '2025-03-07', 'reference_piece': 'SKU-10008', 'designation': 'Batterie 12V', 'qte': '6', 'poids_unitaire_g': '740', 'entrepot': 'OMG-MAR'}
  {'ref_commande': 'ND-001297', 'date_cde': '07/03/2025', 'reference_piece': 'SKU-10008', 'designation': 'Batterie 12V', 'qte': '3', 'poids_unitaire_g': '740', 'entrepot': 'OMG-MAR'}
  {'ref_commande': 'ND-001297', 'date_cde': '07/03/2025', 'reference_piece': 'SKU-10008', 'designation': 'Batterie 12V', 'qte': '3', 'poids_unitaire_g': '740', 'entrepot': 'OMG-MAR'}


Les quantités diffèrent (`qte`) et le format de date change d'une ligne à
l'autre, alors qu'il s'agit très probablement de la même date (juste
reformatée). C'est ce type de conflit que `deduplicate()` résout en
conservant la **dernière occurrence rencontrée** — voir la justification
métier détaillée dans le docstring de cette fonction
(`src/datacore/processing/clients_cleaning.py`) : hypothèse d'une mise à
jour de commande par le client, non prouvable à partir des seules
données disponibles, et donc comptabilisée comme conflit résolu plutôt
que silencieusement écrasée.


## 3. Valeurs manquantes


In [6]:
for name, (path, sep, key, prod_key) in FILES.items():
    rows = rows_by_client[name]
    print(f"--- {name} ---")
    for col in rows[0].keys():
        n_empty = sum(1 for r in rows if not r[col] or not r[col].strip())
        if n_empty:
            print(f"  {col}: {n_empty} valeurs vides")


--- norddrive ---
  qte: 29 valeurs vides
--- freshmarket ---
  quantite_commandee: 27 valeurs vides
--- mediotex ---


Seule la colonne quantité est concernée (29 chez NordDrive, 27 chez
FreshMarket, 0 chez MedioTex) — ces lignes sont traitées comme des
entrées corrompues et écartées par `normalize_norddrive`/`normalize_freshmarket`
(impossible de normaliser une ligne sans quantité).


## 4. Formats de date rencontrés

Détection par remplacement des chiffres par `#`, pour faire apparaître le
motif de chaque date sans interpréter sa valeur.


In [7]:
import re

DATE_COLS = {"norddrive": "date_cde", "freshmarket": "date_reception", "mediotex": "date"}

for name, col in DATE_COLS.items():
    rows = rows_by_client[name]
    patterns = Counter(re.sub(r"\d", "#", r[col]) for r in rows)
    print(f"--- {name} ({col}) ---")
    for p, c in patterns.most_common():
        print(" ", p, c)


--- norddrive (date_cde) ---
  ##/##/#### 525
  ##-##-#### 483
  ####-##-## 471
--- freshmarket (date_reception) ---
  ##/##/#### 464
  ##-##-#### 412
  ####-##-## 412
--- mediotex (date) ---
  ##/##/#### 512
  ####-##-## 501
  ##-##-#### 483


Trois formats coexistent dans les trois fichiers : `DD/MM/YYYY`,
`DD-MM-YYYY` et `YYYY-MM-DD` — la première version de la topographie des
données n'en documentait que deux. `parse_date()` les gère tous les
trois.

### Confirmation qu'il s'agit bien de DD/MM/YYYY (et non MM/DD/YYYY)

On cherche une ligne où le premier nombre dépasse 12 : impossible pour un
mois, ce qui confirme la convention française jour/mois/année.


In [8]:
rows = rows_by_client["norddrive"]
for r in rows:
    v = r["date_cde"]
    if "/" in v:
        d, m, y = v.split("/")
        if int(d) > 12:
            print("Confirmé DD/MM/YYYY, exemple :", v)
            break


Confirmé DD/MM/YYYY, exemple : 16/05/2026


## 5. Cohérence des valeurs métier

Vérification que les codes entrepôt et le booléen métier ne comportent
pas de valeurs aberrantes ou de variantes orthographiques.


In [9]:
fm = rows_by_client["freshmarket"]
print("chaine_froid_requise :", set(r["chaine_froid_requise"] for r in fm))
print("site_livraison (FreshMarket) :", set(r["site_livraison"] for r in fm))

nd = rows_by_client["norddrive"]
print("entrepot (NordDrive) :", set(r["entrepot"] for r in nd))

mtx = rows_by_client["mediotex"]
print("entrepot_destination (MedioTex) :", set(r["entrepot_destination"] for r in mtx))


chaine_froid_requise : {'OUI'}
site_livraison (FreshMarket) : {'OMG-MAR', 'OMG-LYO', 'OMG-LIL'}
entrepot (NordDrive) : {'OMG-MAR', 'OMG-LYO', 'OMG-LIL'}
entrepot_destination (MedioTex) : {'OMG-MAR', 'OMG-LYO', 'OMG-LIL'}


Aucune anomalie : les codes entrepôt correspondent exactement aux trois
sites connus (`OMG-LYO`, `OMG-LIL`, `OMG-MAR`, cohérents avec
`entrepots.code` de FluxPro), et le booléen métier n'a pas de variante
orthographique. Aucun nettoyage supplémentaire n'est nécessaire sur ces
colonnes.

## Conclusion

Les chiffres retenus dans `docs/architecture/topographie_donnees.md` §3.3
et implémentés dans `src/datacore/processing/clients_cleaning.py`
découlent directement de cette exploration :

- grain de mesure des doublons : **(commande, produit)**, pas commande seule ;
- taux de vrais doublons : **13 à 17 %** selon le client ;
- valeurs manquantes : uniquement sur la quantité (29 / 27 / 0) ;
- 3 formats de date à gérer : `DD/MM/YYYY`, `DD-MM-YYYY`, `YYYY-MM-DD` ;
- codes entrepôt et booléen métier propres, sans nettoyage requis.
